# Agent: Observer Agent (Sensor Network)

**Purpose:** Deploy water-level sensors and publish readings every 2 seconds.

**Sensors:** 5 sensors deployed around Køge Søndre Strand (55.4506, 12.1975)

**Publishes to:** `city/flood/observer/sensor_1` through `sensor_5` topics as JSON `ObserverReading` messages

**Data:** Water level (meters) + flow rate (simulated)

In [45]:
import time
import json
import random
from datetime import datetime, timezone
from pathlib import Path
import sys

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import ObserverReading

# Load configuration
cfg = load_config()
mqtt_cfg = cfg.mqtt
flood_cfg = cfg.flood

print(f"Configuration loaded:")
print(f"  MQTT Broker: {mqtt_cfg.host}:{mqtt_cfg.port}")
print(f"  Base Topic: {mqtt_cfg.base_topic}")
print(f"  Flood config: control_threshold={flood_cfg.control_threshold}m")

# Define sensor locations (around Køge Søndre Strand)
BASE_LAT, BASE_LNG = 55.4506, 12.1975
num_sensors = 5
sensors = {
    f"sensor_{i+1}": (BASE_LAT + random.uniform(-0.0005, 0.0005), 
                      BASE_LNG + random.uniform(-0.0005, 0.0005))
    for i in range(num_sensors)
}

# Flood cycle configuration
BASELINE_WATER = 0.2
RISE_RATE = 0.20
RISE_DURATION_S = 30.0
FALL_DURATION_S = 15.0
CYCLE_DURATION_S = RISE_DURATION_S + FALL_DURATION_S
MAX_FLOOD_WATER = BASELINE_WATER + (RISE_RATE * RISE_DURATION_S)  # 6.2m

# Global cycle state
global_water_level = BASELINE_WATER
cycle_elapsed_s = 0.0
current_phase = "rising"

print("\nObserver network configured:")
print(f"  Base location: Køge Søndre Strand ({BASE_LAT}, {BASE_LNG})")
print(f"  Sensors: {num_sensors}")
print(f"  Cycle: rise {RISE_DURATION_S:.0f}s at {RISE_RATE:.2f}m/s, fall {FALL_DURATION_S:.0f}s")
for sensor_id, (lat, lng) in sensors.items():
    print(f"    - {sensor_id}: ({lat:.4f}, {lng:.4f})")

Configuration loaded:
  MQTT Broker: 127.0.0.1:1883
  Base Topic: simulated-city
  Flood config: control_threshold=1.0m

Observer network configured:
  Base location: Køge Søndre Strand (55.4506, 12.1975)
  Sensors: 5
  Cycle: rise 30s at 0.20m/s, fall 15s
    - sensor_1: (55.4507, 12.1973)
    - sensor_2: (55.4503, 12.1977)
    - sensor_3: (55.4508, 12.1975)
    - sensor_4: (55.4504, 12.1978)
    - sensor_5: (55.4510, 12.1974)


In [46]:
# Connect to MQTT broker
connector = MqttConnector(mqtt_cfg, client_id_suffix='observer')
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
    publisher = MqttPublisher(connector)
else:
    print("✗ Failed to connect to MQTT broker")
    publisher = None

✓ Connected to MQTT broker


In [47]:
def update_global_water_level(elapsed_s: float) -> float:
    """Update water level on repeating cycle: rise 30s, fall 15s."""
    global global_water_level, cycle_elapsed_s, current_phase

    cycle_position = cycle_elapsed_s % CYCLE_DURATION_S
    
    if cycle_position < RISE_DURATION_S:
        # Rising phase: increase by RISE_RATE per second
        global_water_level = BASELINE_WATER + (RISE_RATE * cycle_position)
        current_phase = "rising"
    else:
        # Falling phase: decrease back to baseline
        fall_position = cycle_position - RISE_DURATION_S
        global_water_level = MAX_FLOOD_WATER - (RISE_RATE * fall_position)
        current_phase = "falling"
    
    global_water_level = max(BASELINE_WATER, min(MAX_FLOOD_WATER, global_water_level))
    cycle_elapsed_s += elapsed_s
    return global_water_level


def publish_sensor_reading(sensor_id: str, lat: float, lng: float) -> ObserverReading:
    """Publish a sensor reading."""
    if publisher is None:
        print(f"  {sensor_id}: MQTT not connected, skipping")
        return None
    
    water_level = global_water_level + random.uniform(-0.05, 0.05)  # ±50mm noise
    flow_rate = max(0, water_level - BASELINE_WATER) * 2.0  # Proportional to height
    
    reading = ObserverReading(
        sensor_id=sensor_id,
        water_level=max(0, water_level),
        flow_rate=flow_rate,
        timestamp=datetime.now(timezone.utc).isoformat()
    )
    
    topic = make_topic(mqtt_cfg, "flood", "observer", sensor_id)
    payload = json.dumps(reading.to_json())
    # Use QoS 0 for faster publishing, no acknowledgment required
    publisher.publish_json(topic, payload, qos=0)
    
    return reading

In [ ]:
# Main sensor network loop
print("\n📊 Starting observer network (publishing every 2 seconds)...")
print(f"Control threshold: {flood_cfg.control_threshold}m")
print("Press Ctrl+C to stop\n")

reading_count = 0
try:
    tick = 0
    
    while True:
        # Update cycle every second
        update_global_water_level(1.0)
        
        # Publish readings every 2 seconds
        tick += 1
        if tick % 2 == 0:
            reading_count += 1
            timestamp = datetime.now(timezone.utc).strftime('%H:%M:%S')
            alert = " ⚠️ ALERT!" if global_water_level >= flood_cfg.control_threshold else ""
            print(f"[{timestamp}] Reading {reading_count}: Phase={current_phase:7s} | Water={global_water_level:.2f}m{alert}")
            
            # Publish all sensor readings
            for sensor_id, (lat, lng) in sensors.items():
                try:
                    publish_sensor_reading(sensor_id, lat, lng)
                except Exception as e:
                    print(f"  Error publishing {sensor_id}: {e}")
        
        time.sleep(1)
        
except KeyboardInterrupt:
    print("\n\n✓ Observer agent stopped by user")
finally:
    if connector:
        connector.disconnect()
        print("MQTT connection closed")


📊 Starting observer network (publishing every 2 seconds)...
Control threshold: 1.0m
Press Ctrl+C to stop

[13:26:03] Reading 1: Phase=rising  | Water=0.40m
[13:26:05] Reading 2: Phase=rising  | Water=0.80m
[13:26:07] Reading 3: Phase=rising  | Water=1.20m ⚠️ ALERT!
[13:26:09] Reading 4: Phase=rising  | Water=1.60m ⚠️ ALERT!
[13:26:11] Reading 5: Phase=rising  | Water=2.00m ⚠️ ALERT!
[13:26:13] Reading 6: Phase=rising  | Water=2.40m ⚠️ ALERT!
[13:26:15] Reading 7: Phase=rising  | Water=2.80m ⚠️ ALERT!
[13:26:17] Reading 8: Phase=rising  | Water=3.20m ⚠️ ALERT!
[13:26:19] Reading 9: Phase=rising  | Water=3.60m ⚠️ ALERT!
[13:26:21] Reading 10: Phase=rising  | Water=4.00m ⚠️ ALERT!
[13:26:23] Reading 11: Phase=rising  | Water=4.40m ⚠️ ALERT!
[13:26:25] Reading 12: Phase=rising  | Water=4.80m ⚠️ ALERT!
[13:26:27] Reading 13: Phase=rising  | Water=5.20m ⚠️ ALERT!
[13:26:29] Reading 14: Phase=rising  | Water=5.60m ⚠️ ALERT!
[13:26:31] Reading 15: Phase=rising  | Water=6.00m ⚠️ ALERT!
[13:26:

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[15:31:39] Reading 104: Phase=rising  | Water=5.60m ⚠️ ALERT!


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...


[17:30:42] Reading 105: Phase=rising  | Water=6.00m ⚠️ ALERT!
[17:30:44] Reading 106: Phase=falling | Water=6.00m ⚠️ ALERT!
[17:30:46] Reading 107: Phase=falling | Water=5.60m ⚠️ ALERT!
[17:30:48] Reading 108: Phase=falling | Water=5.20m ⚠️ ALERT!
[17:30:50] Reading 109: Phase=falling | Water=4.80m ⚠️ ALERT!
[17:30:52] Reading 110: Phase=falling | Water=4.40m ⚠️ ALERT!
[17:30:54] Reading 111: Phase=falling | Water=4.00m ⚠️ ALERT!
[17:30:56] Reading 112: Phase=falling | Water=3.60m ⚠️ ALERT!
[17:30:58] Reading 113: Phase=rising  | Water=0.20m
[17:31:00] Reading 114: Phase=rising  | Water=0.60m
[17:31:02] Reading 115: Phase=rising  | Water=1.00m ⚠️ ALERT!
[17:31:04] Reading 116: Phase=rising  | Water=1.40m ⚠️ ALERT!
[17:31:06] Reading 117: Phase=rising  | Water=1.80m ⚠️ ALERT!
[17:31:08] Reading 118: Phase=rising  | Water=2.20m ⚠️ ALERT!
[17:31:10] Reading 119: Phase=rising  | Water=2.60m ⚠️ ALERT!
[17:31:12] Reading 120: Phase=rising  | Water=3.00m ⚠️ ALERT!
[17:31:14] Reading 121: Phas

...

[20:05:18] Reading 1212: Phase=falling | Water=4.60m ⚠️ ALERT!
[20:05:20] Reading 1213: Phase=falling | Water=4.20m ⚠️ ALERT!
[20:05:22] Reading 1214: Phase=falling | Water=3.80m ⚠️ ALERT!
[20:05:24] Reading 1215: Phase=falling | Water=3.40m ⚠️ ALERT!
[20:05:27] Reading 1216: Phase=rising  | Water=0.40m
[20:05:30] Reading 1217: Phase=rising  | Water=0.80m
[20:05:32] Reading 1218: Phase=rising  | Water=1.20m ⚠️ ALERT!
[20:05:34] Reading 1219: Phase=rising  | Water=1.60m ⚠️ ALERT!
[20:05:36] Reading 1220: Phase=rising  | Water=2.00m ⚠️ ALERT!
[20:05:38] Reading 1221: Phase=rising  | Water=2.40m ⚠️ ALERT!
[20:05:40] Reading 1222: Phase=rising  | Water=2.80m ⚠️ ALERT!
[20:05:43] Reading 1223: Phase=rising  | Water=3.20m ⚠️ ALERT!
[20:05:45] Reading 1224: Phase=rising  | Water=3.60m ⚠️ ALERT!
[20:05:47] Reading 1225: Phase=rising  | Water=4.00m ⚠️ ALERT!
[20:05:49] Reading 1226: Phase=rising  | Water=4.40m ⚠️ ALERT!
[20:05:51] Reading 1227: Phase=rising  | Water=4.80m ⚠️ ALERT!
[20:05:53] R

Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
MQTT client not connected. Message may not be published.
MQTT client not connected. Message may not be published.
MQTT client not connected. Message may not be published.
MQTT client not connected. Message may not be published.


[21:49:57] Reading 1945: Phase=rising  | Water=4.00m ⚠️ ALERT!


Disconnected from MQTT broker (reason=Unspecified error). Reconnecting...
MQTT client not connected. Message may not be published.
MQTT client not connected. Message may not be published.
MQTT client not connected. Message may not be published.
MQTT client not connected. Message may not be published.
MQTT client not connected. Message may not be published.


[22:42:27] Reading 1946: Phase=rising  | Water=4.40m ⚠️ ALERT!
[22:42:29] Reading 1947: Phase=rising  | Water=4.80m ⚠️ ALERT!
[22:42:31] Reading 1948: Phase=rising  | Water=5.20m ⚠️ ALERT!
[22:42:33] Reading 1949: Phase=rising  | Water=5.60m ⚠️ ALERT!
[22:42:36] Reading 1950: Phase=rising  | Water=6.00m ⚠️ ALERT!
[22:42:38] Reading 1951: Phase=falling | Water=6.00m ⚠️ ALERT!
[22:42:40] Reading 1952: Phase=falling | Water=5.60m ⚠️ ALERT!
[22:42:42] Reading 1953: Phase=falling | Water=5.20m ⚠️ ALERT!
[22:42:44] Reading 1954: Phase=falling | Water=4.80m ⚠️ ALERT!
[22:42:46] Reading 1955: Phase=falling | Water=4.40m ⚠️ ALERT!
[22:42:48] Reading 1956: Phase=falling | Water=4.00m ⚠️ ALERT!
[22:42:50] Reading 1957: Phase=falling | Water=3.60m ⚠️ ALERT!


In [ ]:
# Test: Publish one reading to verify MQTT connection
print("Testing MQTT connection with a single reading...")
test_reading = publish_sensor_reading("sensor_1", BASE_LAT, BASE_LNG)
if test_reading:
    print(f"\n✓ Test successful!")
    print(f"  Sensor {test_reading.sensor_id} published to MQTT")
    print(f"  Water level: {test_reading.water_level:.2f}m")
    print(f"  Flow rate: {test_reading.flow_rate:.2f} m³/s")
    print(f"  Topic: simulated-city/flood/observer/sensor_1")
else:
    print("\n✗ Test failed - MQTT not connected")

Testing MQTT connection with a single reading...

✓ Test successful!
  Sensor sensor_1 published to MQTT
  Water level: 0.18m
  Flow rate: 0.00 m³/s
  Topic: simulated-city/flood/observer/sensor_1
